# C3 domain-readability diagnostic (the C4 gate)

**Question:** do SupCon features retain the domain information that CE
features discard? The C1/C2 sessions showed a plain CE encoder keeps no
linearly readable domain at all, so the GRL had nothing to remove. SupCon
compresses less than CE by construction — if the domain IS readable from
C3's features, the C4 GRL has a live target and C4 is worth launching;
if it sits at the majority baselines here too, the GRL branch closes with
evidence on both loss families.

**Run AFTER phase B** (notebook `04`, cell 4, `c3_supcon`): this session
reuses the feature caches phase B creates next to the grid checkpoints
(`features_epoch40/50/60_train.npz`) and reads `phase_b_selection.json`.
No data staging, no encoder forward passes. GPU speeds the probes up but
CPU works.

Same instrument as the C1/C2 sessions (identical recipe, seed, inner
split — deterministic from the frozen train trace list), extended with a
**maturity sweep**: the diagnostic runs on all three grid checkpoints,
not just the selected one, to test the compression hypothesis (younger
checkpoints retain more input information — cf. C2@13 vs C1@37).

Train/val features only — no test contact (§0.7). Declared confound: the
encoder trained on all 81 train traces, so absolute numbers are inflated
by memorization; contrasts across checkpoints and against C1/C2 stay fair
(same traces everywhere).

## Environment setup

Reduced preamble: locate/clone the repo, install requirements, mount
Drive. No data staging — this session runs on cached features only.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

drive.mount("/content/drive")  # idempotent
CKPT_ROOT = Path(read_yaml(REPO_DIR / "configs" / "paths.yaml")["ckpt_root"])
print("Repo dir:", REPO_DIR)
print("Checkpoint root:", CKPT_ROOT)

## Diagnostic

Identical code to the C1/C2 sessions (`2026-07-17_c1_ce_domain_probe`,
`2026-07-17_c2_grl_domain_probe`), with the cache stem parametrized —
C3 has no `best.ckpt` (§6-C3), its checkpoints are the phase-B grid.
Imports the frozen §5.3 recipe; defines nothing in `sharp_har/`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from sharp_har.harness import fuse_windows
from sharp_har.inventory import AR_SET_METADATA
from sharp_har.probe import linear_probe, majority_baseline

RUN = "C3"
SEED = 42          # §0 rule 5
EVAL_FRAC = 1 / 3  # identical inner split to the C1/C2 sessions
TARGETS = ("y", "ar_set", "ambiente", "direct_path", "persona", "monitor")

# Reference rows from the archived C1/C2 sessions (acc, baseline) —
# printed alongside for side-by-side reading, never recomputed here.
REFERENCE = {
    ("C1", "y"): (1.000, 0.197), ("C2", "y"): (0.893, 0.197),
    ("C1", "ar_set"): (0.196, 0.286), ("C2", "ar_set"): (0.257, 0.286),
    ("C1", "ambiente"): (0.854, 0.854), ("C2", "ambiente"): (0.854, 0.854),
    ("C1", "direct_path"): (0.633, 0.731), ("C2", "direct_path"): (0.739, 0.731),
    ("C1", "persona"): (0.818, 0.818), ("C2", "persona"): (0.818, 0.818),
    ("C1", "monitor"): (0.499, 0.584), ("C2", "monitor"): (0.600, 0.584),
}


def build_targets(arset_name, arset_int, arset_labels) -> dict:
    """ar_set (the adversary's target in C2/C4) + the metadata attributes
    it conflates. The ar_set label space comes from the cache's own
    arset_labels; derived spaces from the AR-sets observed here (no class
    for AR-7 — test, absent). Identical across runs: frozen trace list."""
    names = np.array([str(a) for a in arset_name])
    ar_labels = [str(a) for a in arset_labels]
    assert int(arset_int.max()) < len(ar_labels), "ar_set index outside arset_labels"
    assert set(names) <= set(ar_labels), "arset_name outside the train-domain label set"
    out = {"ar_set": (arset_int.astype(np.int64), ar_labels)}
    for attr in ("ambiente", "direct_path", "persona", "monitor"):
        vals = sorted({AR_SET_METADATA[n][attr] for n in set(names)})
        out[attr] = (
            np.array([vals.index(AR_SET_METADATA[n][attr]) for n in names], dtype=np.int64),
            vals,
        )
    return out


def inner_split(trace_id, arset_int, seed: int = SEED):
    """Trace-disjoint split stratified by AR-set (§0 rule 2: never by
    window). Deterministic: sorted traces, one seeded permutation per
    AR-set — bit-identical to the C1/C2 sessions."""
    trace_to_ar: dict[str, int] = {}
    for t, a in zip(trace_id, arset_int):
        trace_to_ar.setdefault(str(t), int(a))
    by_ar: dict[int, list[str]] = {}
    for t, a in trace_to_ar.items():
        by_ar.setdefault(a, []).append(t)
    rng = np.random.default_rng(seed)
    fit, evl = [], []
    for a in sorted(by_ar):
        ts = np.array(sorted(by_ar[a]), dtype=object)
        ts = ts[rng.permutation(len(ts))]
        n_ev = max(1, int(round(len(ts) * EVAL_FRAC)))
        evl += list(ts[:n_ev])
        fit += list(ts[n_ev:])
    fit_s, ev_s = set(fit), set(evl)
    assert not (fit_s & ev_s), "inner split is not trace-disjoint (§0 rule 2)"
    assert fit_s and ev_s
    return fit_s, ev_s


def fused_accuracy(x, y, tid, ws, weight, bias):
    """Antenna-fused accuracy of the probe head + the majority baseline
    on the SAME fused units (§7 reads accuracy vs majority, not 1/n)."""
    logits = x @ weight.T + bias
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = (np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)).astype(np.float32)
    fused = fuse_windows(probs, y, tid.astype(object), ws)
    acc = float((fused["y_pred"] == fused["y_true"]).mean())
    return acc, majority_baseline(fused["y_true"])


def diagnose(run_dir: Path, label: str, stem: str) -> list[dict]:
    path = run_dir / f"features_{stem}_train.npz"
    if not path.exists():
        print(f"[{label}] SKIP - {path} not found (phase B caches it; run notebook 04 cell 4 first)")
        return []
    d = np.load(path, allow_pickle=False)
    x, tid, ws = d["features"], d["trace_id"], d["window_start"]
    targets = build_targets(d["arset_name"], d["ar_set"], d["arset_labels"])
    targets["y"] = (d["y"].astype(np.int64), [str(l) for l in d["labels"]])
    fit_s, ev_s = inner_split(tid, d["ar_set"])
    m_fit = np.array([str(t) in fit_s for t in tid])
    m_ev = np.array([str(t) in ev_s for t in tid])
    print(f"\n[{label}] {path.name}: {x.shape[0]} samples, d={x.shape[1]}")
    print(f"[{label}] inner split: {len(fit_s)} fit / {len(ev_s)} eval traces "
          f"({m_fit.sum()} / {m_ev.sum()} samples)")
    rows = []
    for target in TARGETS:
        y, vals = targets[target]
        n_cls = len(vals)
        fit_d = {"features": x[m_fit], target: y[m_fit], "trace_id": tid[m_fit], "window_start": ws[m_fit]}
        ev_d = {"features": x[m_ev], target: y[m_ev], "trace_id": tid[m_ev], "window_start": ws[m_ev]}
        res = linear_probe(fit_d, ev_d, target=target, n_classes=n_cls, seed=SEED)
        acc, base = fused_accuracy(x[m_ev], y[m_ev], tid[m_ev], ws[m_ev], res["weight"], res["bias"])
        rows.append({"run": label, "target": target, "eval_accuracy": acc,
                     "majority_baseline": base, "delta": acc - base,
                     "macro_f1": res["best_val_macro_f1"], "best_epoch": res["best_epoch"]})
        tag = "  <- CONTROL" if target == "y" else ""
        print(f"[{label}] {target:12s} ({n_cls} cls): acc {acc:.3f} vs baseline {base:.3f} "
              f"(delta {acc - base:+.3f}), macro-F1 {res['best_val_macro_f1']:.3f}{tag}")
    return rows

In [ ]:
sel_path = CKPT_ROOT / RUN / "phase_b_selection.json"
assert sel_path.exists(), (
    f"{sel_path} not found - run phase B first (notebook 04, cell 4, c3_supcon): "
    "this diagnostic reuses the feature caches phase B creates."
)
sel = json.loads(sel_path.read_text())
print(f"phase B selection: epoch {sel['selected_epoch']} "
      f"(val macro-F1 {sel['selected_val_macro_f1']:.4f}, 5-class val - see split audit)")
for c in sel["candidates"]:
    print(f"  candidate epoch {c['epoch']}: val macro-F1 {c['best_val_macro_f1']:.4f}")

# Maturity sweep: the diagnostic on EVERY grid checkpoint (caches exist
# after phase B), selected one marked. Tests the compression hypothesis:
# younger checkpoints should read domain more easily, if it is there at all.
rows = []
for c in sel["candidates"]:
    stem = f"epoch{c['epoch']}"
    mark = " (selected)" if c["epoch"] == sel["selected_epoch"] else ""
    rows += diagnose(CKPT_ROOT / RUN, f"{RUN}@{stem}{mark}", stem=stem)

print("\n" + "=" * 78)
print(f"{'run':22s} {'target':12s} {'acc':>7s} {'baseline':>9s} {'delta':>8s} {'macroF1':>8s}")
print("-" * 78)
for r in rows:
    print(f"{r['run']:22s} {r['target']:12s} {r['eval_accuracy']:7.3f} "
          f"{r['majority_baseline']:9.3f} {r['delta']:+8.3f} {r['macro_f1']:8.3f}")
print("-" * 78)
print("reference (archived C1/C2 sessions, same instrument):")
for (run, target), (acc, base) in REFERENCE.items():
    print(f"{run:22s} {target:12s} {acc:7.3f} {base:9.3f} {acc - base:+8.3f}")
print("=" * 78)
print("\nREAD THE CONTROL FIRST: y must be high with a large positive delta")
print("(C1 scored 1.000 on these seen traces). If not, stop.")
print("\nThen the C4 gate: domain deltas >> 0 (unlike C1/C2) => SupCon retains")
print("domain, the C4 GRL has a live target, C4 launches as planned. Deltas at")
print("~0 here too => no readable domain under either loss family, the GRL")
print("branch closes. Maturity trend across epoch40/50/60 is the secondary")
print("reading (compression hypothesis). Team call either way - findings go")
print("to STATUS.md, this notebook is the evidence.")

## Archiving

Download the executed copy with outputs and commit it verbatim over this
file (same name), then fill the Outcome column of this session's row in
`notebooks/diagnostics/README.md` and add the STATUS line — same commit.